<a href="https://colab.research.google.com/github/anacaetano02/Projetos-Redes-Neurais/blob/main/Projeto_1_MLP_src_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.makedirs('src', exist_ok=True)

In [2]:
%%writefile src/model.py

"""
model.py — Definições das arquiteturas MLP para classificação e regressão.
Dataset: Lending Club (Kaggle)
Autor: MBA Engenharia de IA
"""

import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# Bloco auxiliar: camada densa com normalização e ativação opcionais
# ---------------------------------------------------------------------------
class CamadaDensa(nn.Module):
    """
    Bloco reutilizável: Linear → (Norm) → Ativação → (Dropout).

    Parâmetros
    ----------
    entrada : int
        Dimensão de entrada.
    saida : int
        Dimensão de saída.
    ativacao : nn.Module
        Instância da função de ativação (ex: nn.ReLU()).
    norm : str ou None
        'batch' para BatchNorm1d, 'layer' para LayerNorm, None para nenhuma.
    dropout_p : float
        Probabilidade de dropout (0.0 = desativado).
    """

    def __init__(self, entrada: int, saida: int,
                 ativacao: nn.Module = None,
                 norm: str = None,
                 dropout_p: float = 0.0):
        super().__init__()

        camadas = [nn.Linear(entrada, saida)]

        if norm == 'batch':
            camadas.append(nn.BatchNorm1d(saida))
        elif norm == 'layer':
            camadas.append(nn.LayerNorm(saida))

        if ativacao is not None:
            camadas.append(ativacao)

        if dropout_p > 0.0:
            camadas.append(nn.Dropout(p=dropout_p))

        self.bloco = nn.Sequential(*camadas)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.bloco(x)


# ---------------------------------------------------------------------------
# MLP para Classificação Binária
# ---------------------------------------------------------------------------
class MLPClassificador(nn.Module):
    """
    MLP para classificação binária (Lending Club: Fully Paid vs Charged Off).

    Arquitetura justificada:
    - Profundidade 3 camadas ocultas: suficiente para capturar interações
      não-lineares entre features financeiras (dti, fico, int_rate, etc.)
      sem risco excessivo de overfitting para ~350k amostras.
    - Largura decrescente (256→128→64): pirâmide de compressão progressiva
      que força representações cada vez mais abstratas do risco de crédito.

    Parâmetros
    ----------
    n_features : int
        Número de features de entrada após pré-processamento.
    camadas_ocultas : list[int]
        Lista com o tamanho de cada camada oculta.
        Padrão: [256, 128, 64]
    ativacao : str
        'relu', 'tanh', 'leaky_relu', 'elu' ou 'selu'.
    norm : str ou None
        'batch', 'layer' ou None.
    dropout_p : float
        Probabilidade de dropout nas camadas ocultas.
    """

    ATIVACOES = {
        'relu':       nn.ReLU(),
        'tanh':       nn.Tanh(),
        'leaky_relu': nn.LeakyReLU(negative_slope=0.01),
        'elu':        nn.ELU(),
        'selu':       nn.SELU(),
    }

    def __init__(self,
                 n_features: int,
                 camadas_ocultas: list = None,
                 ativacao: str = 'relu',
                 norm: str = None,
                 dropout_p: float = 0.0):
        super().__init__()

        if camadas_ocultas is None:
            camadas_ocultas = [256, 128, 64]

        if ativacao not in self.ATIVACOES:
            raise ValueError(f"Ativação '{ativacao}' não suportada. "
                             f"Escolha: {list(self.ATIVACOES.keys())}")

        # Constrói as camadas ocultas dinamicamente
        blocos = []
        dim_entrada = n_features
        for dim_saida in camadas_ocultas:
            # Cria nova instância da ativação para cada bloco (evita estado compartilhado)
            ativ = self._nova_ativacao(ativacao)
            blocos.append(CamadaDensa(dim_entrada, dim_saida,
                                      ativacao=ativ,
                                      norm=norm,
                                      dropout_p=dropout_p))
            dim_entrada = dim_saida

        self.camadas_ocultas = nn.Sequential(*blocos)

        # Camada de saída: sem ativação (BCEWithLogitsLoss espera logits)
        self.saida = nn.Linear(dim_entrada, 1)

        # Salva hiperparâmetros para inspeção
        self.n_features = n_features
        self.config_ativacao = ativacao
        self.config_norm = norm
        self.config_dropout = dropout_p

    def _nova_ativacao(self, nome: str) -> nn.Module:
        """Retorna nova instância para evitar reutilização de estado."""
        mapa = {
            'relu':       nn.ReLU(),
            'tanh':       nn.Tanh(),
            'leaky_relu': nn.LeakyReLU(negative_slope=0.01),
            'elu':        nn.ELU(),
            'selu':       nn.SELU(),
        }
        return mapa[nome]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Entrada: (batch, n_features)
        Saída:   (batch, 1) — logit não normalizado
        """
        x = self.camadas_ocultas(x)
        return self.saida(x)

    def resumo(self) -> str:
        """Retorna string descritiva da arquitetura."""
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return (f"MLPClassificador | Ativação: {self.config_ativacao} | "
                f"Norm: {self.config_norm} | Dropout: {self.config_dropout} | "
                f"Parâmetros treináveis: {total_params:,}")


# ---------------------------------------------------------------------------
# MLP para Regressão
# ---------------------------------------------------------------------------
class MLPRegressor(nn.Module):
    """
    MLP para regressão: prediz int_rate (taxa de juros) como variável contínua.

    Arquitetura:
    - Mesma lógica da pirâmide do classificador, mas a saída é um escalar
      contínuo (sem sigmoid/softmax).
    - Loss: MSELoss | Métricas: MAE, RMSE, R²

    Parâmetros
    ----------
    n_features : int
        Número de features de entrada (mesmo conjunto do classificador,
        mas sem int_rate que agora é o target).
    camadas_ocultas : list[int]
        Padrão: [256, 128, 64]
    ativacao : str
        Mesmas opções do MLPClassificador.
    norm : str ou None
        'batch', 'layer' ou None.
    dropout_p : float
        Probabilidade de dropout.
    """

    def __init__(self,
                 n_features: int,
                 camadas_ocultas: list = None,
                 ativacao: str = 'relu',
                 norm: str = None,
                 dropout_p: float = 0.0):
        super().__init__()

        if camadas_ocultas is None:
            camadas_ocultas = [256, 128, 64]

        blocos = []
        dim_entrada = n_features
        for dim_saida in camadas_ocultas:
            ativ = self._nova_ativacao(ativacao)
            blocos.append(CamadaDensa(dim_entrada, dim_saida,
                                      ativacao=ativ,
                                      norm=norm,
                                      dropout_p=dropout_p))
            dim_entrada = dim_saida

        self.camadas_ocultas = nn.Sequential(*blocos)

        # Saída escalar contínua (sem ativação)
        self.saida = nn.Linear(dim_entrada, 1)

        self.n_features = n_features
        self.config_ativacao = ativacao
        self.config_norm = norm
        self.config_dropout = dropout_p

    def _nova_ativacao(self, nome: str) -> nn.Module:
        mapa = {
            'relu':       nn.ReLU(),
            'tanh':       nn.Tanh(),
            'leaky_relu': nn.LeakyReLU(negative_slope=0.01),
            'elu':        nn.ELU(),
            'selu':       nn.SELU(),
        }
        return mapa[nome]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Entrada: (batch, n_features)
        Saída:   (batch, 1) — valor contínuo (int_rate previsto)
        """
        x = self.camadas_ocultas(x)
        return self.saida(x)

    def resumo(self) -> str:
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return (f"MLPRegressor | Ativação: {self.config_ativacao} | "
                f"Norm: {self.config_norm} | Dropout: {self.config_dropout} | "
                f"Parâmetros treináveis: {total_params:,}")


# ---------------------------------------------------------------------------
# Funções de inicialização de pesos
# ---------------------------------------------------------------------------
def inicializar_pesos(modelo: nn.Module, metodo: str = 'xavier_uniform') -> None:
    """
    Aplica inicialização de pesos a todas as camadas Linear do modelo.

    Parâmetros
    ----------
    modelo : nn.Module
        Modelo a ser inicializado.
    metodo : str
        'xavier_uniform'  — Xavier/Glorot uniforme (bom para tanh/sigmoid)
        'xavier_normal'   — Xavier/Glorot normal
        'he_uniform'      — He uniforme (bom para ReLU e variantes)
        'he_normal'       — He normal (padrão do PyTorch para ReLU)
        'padrao'          — Mantém a inicialização padrão do PyTorch (Kaiming uniform)
    """
    if metodo == 'padrao':
        # Não faz nada — mantém o padrão do PyTorch
        return

    for modulo in modelo.modules():
        if isinstance(modulo, nn.Linear):
            if metodo == 'xavier_uniform':
                nn.init.xavier_uniform_(modulo.weight)
            elif metodo == 'xavier_normal':
                nn.init.xavier_normal_(modulo.weight)
            elif metodo == 'he_uniform':
                nn.init.kaiming_uniform_(modulo.weight, nonlinearity='relu')
            elif metodo == 'he_normal':
                nn.init.kaiming_normal_(modulo.weight, nonlinearity='relu')
            else:
                raise ValueError(f"Método '{metodo}' não reconhecido.")

            if modulo.bias is not None:
                nn.init.zeros_(modulo.bias)


def inspecionar_pesos(modelo: nn.Module) -> None:
    """
    Imprime estatísticas dos pesos de cada camada Linear.
    Útil para comparar distribuições antes/depois da inicialização.
    """
    print(f"\n{'='*60}")
    print(f"{'Camada':<30} {'Média':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print(f"{'='*60}")
    for nome, param in modelo.named_parameters():
        if 'weight' in nome:
            dados = param.data
            print(f"{nome:<30} {dados.mean().item():>10.4f} "
                  f"{dados.std().item():>10.4f} "
                  f"{dados.min().item():>10.4f} "
                  f"{dados.max().item():>10.4f}")
    print(f"{'='*60}\n")


Writing src/model.py
